# ASTR 457: Foundations of Data Science in Astronomy

**Fall 2026, University of Illinois Urbana-Champaign**

Prof. Gautham Narayan | TA: Abha Vishwakarma

Thu Oct 8, 2026 - Day 14: MCMC that you can trust: detailed balance, $\hat R$, HMC

<img src="images/uiuc_logo.png" alt="University of Illinois Urbana-Champaign wordmark" width="220" style="display:block;margin:0 auto;">

## Today's plan

* today's question: how do we know a sampler's answer is right? Three checks:
    1. the chain samples $p$ (detailed balance)
    2. chains from overdispersed starts agree ($\hat R$)
    3. there are enough independent samples (ESS)
* recap: Tuesday's two questions about nested sampling, answered with a picture
* in-class exercise: a line, or a quadratic? Then this week's logistics
* the three checks, one at a time, then HMC and `PyMC`
* two lab defenses at the end

In [ ]:
# Presenter setup (a RISE "skip" cell: it runs, but is not a slide).
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
from scipy.interpolate import BSpline
import arviz as az
plt.rcParams.update({'font.size': 13, 'legend.fontsize': 12})
from ipywidgets import interactive_output, IntSlider, SelectionSlider, Checkbox, HBox, VBox, Layout
from IPython.display import display

def controls_right(fn, **sliders):
    # the plot on the left, its sliders stacked on the right (the Day 12 layout)
    for w in sliders.values():
        w.layout = Layout(width='230px'); w.style = {'description_width': '75px'}
    display(HBox([interactive_output(fn, sliders), VBox(list(sliders.values()), layout=Layout(justify_content='center', width='240px'))]))

# matplotlib's math-text parser is a shared singleton and is not thread-safe. ipykernel 7 handles slider messages on
# another thread (ipykernel #1564), so a redraw can overlap the next cell and break any "$...$" label
# ("Expected end of text", ipympl #610). A lock around the parser fixes that on any kernel version (ipympl PR #621).
import threading
try:
    from matplotlib import _mathtext
    if not getattr(_mathtext.Parser.parse, '_locked', False):
        _mathtext_lock = threading.Lock()
        _unlocked_parse = _mathtext.Parser.parse
        def _locked_parse(self, *args, **kwargs):
            with _mathtext_lock:
                return _unlocked_parse(self, *args, **kwargs)
        _locked_parse._locked = True
        _mathtext.Parser.parse = _locked_parse
except (ImportError, AttributeError):
    print("could not lock matplotlib's math-text parser: do not move a slider while another cell is running")

# The slow results (the R-hat chains, the HMC runs and the PyMC run) are saved in data/day14_cache.npz, so Run All
# takes seconds. Set RECOMPUTE = True to redo them from scratch (about a minute) and rewrite the file.
import os
CACHE_FILE, RECOMPUTE = 'data/day14_cache.npz', False
_cache = {} if RECOMPUTE or not os.path.exists(CACHE_FILE) else dict(np.load(CACHE_FILE, allow_pickle=False))
def cached(name, compute):
    # compute() returns a dict of arrays; they are stored as name/key, loaded back on the next run
    keys = [k for k in _cache if k.startswith(name + '/')]
    if not keys:
        for k, v in compute().items():
            _cache[f'{name}/{k}'] = np.asarray(v)
        np.savez_compressed(CACHE_FILE, **_cache)
        keys = [k for k in _cache if k.startswith(name + '/')]
    return {k.split('/', 1)[1]: _cache[k] for k in keys}

def ess_per_iter(x):
    # effective sample size per iteration: Day 12's N / tau, computed by ArviZ (it can exceed 1)
    return float(az.ess(np.asarray(x)[None, :])) / len(x)

# the in-class exercise's data (same seed as lecture/07/inclass_exercise_day14.ipynb)
_rng = np.random.default_rng(457)
t_ex = np.arange(0, 10, 0.1); sig_ex = 1.0
y_ex = 0.5 * t_ex + 0.2 + _rng.normal(0, sig_ex, t_ex.size)
print(f"exercise data: {t_ex.size} points; setup done")

## Recap

* Tuesday's two questions about nested sampling: how is it different from maximizing? Can local peaks fool it?
* think of the likelihood as mountains and valleys. Nested sampling floods the valley with water, and the water level keeps rising
* the live points are always above the water - the current $L_{\min}$
* slide the iteration to 100, then 150. Watch: the water reaches the lower peak, and its live points move to the higher one

In [ ]:
# Nested sampling on a two-peaked 1-D likelihood, drawn as a flooded landscape (a "skip" cell)
FL_TH = np.linspace(-10, 10, 4001)                                          # the prior: uniform on [-10, 10]
def fl_like(th):
    return 0.55 * np.exp(-0.5 * ((th + 2.5) / 0.9)**2) + 1.0 * np.exp(-0.5 * ((th - 3.0) / 0.5)**2)
FL_L = fl_like(FL_TH)
def _flood(n_live=40, n_iter=170, seed=4):
    rng = np.random.default_rng(seed)
    live = rng.uniform(-10, 10, n_live); levels, lives = [], []
    for k in range(n_iter):
        w = np.argmin(fl_like(live)); lmin = fl_like(live[w])
        levels.append(lmin); lives.append(live.copy())
        ok = FL_TH[FL_L > lmin]                                             # above the water: the prior restricted to L > L_min
        live[w] = rng.choice(ok) + rng.uniform(-0.0025, 0.0025)              # (in 1-D we can draw from it exactly)
    return np.array(levels), np.array(lives)
fl_levels, fl_lives = _flood()
def plot_flood(iteration=20):
    lmin, live = fl_levels[iteration], fl_lives[iteration]
    fig, ax = plt.subplots(figsize=(11, 3.7))
    ax.fill_between(FL_TH, FL_L, lmin, where=FL_L < lmin, color='C0', alpha=0.35, lw=0, label='water: $L < L_{\\min}$')
    ax.plot(FL_TH, FL_L, 'k', lw=2, label='likelihood $L(\\theta)$')
    ax.axhline(lmin, color='C0', lw=1.2)
    ax.plot(live, fl_like(live), 'o', color='C3', ms=6, label='live points')
    nl, nr = np.sum(live < 0.5), np.sum(live >= 0.5)
    ax.set_title(f'iteration {iteration}: water level {lmin:.2f};  live points on the left peak {nl}, right peak {nr}', fontsize=12)
    ax.set_xlabel(r'$\theta$ (prior: uniform from $-10$ to $10$)'); ax.set_ylabel('likelihood'); ax.set_xlim(-6, 6); ax.set_ylim(0, 1.12)
    ax.legend(frameon=False, loc='upper left', fontsize=11)
    plt.tight_layout(); plt.show()

In [ ]:
# black: a likelihood with two peaks; blue: the water, below L_min; red: the 40 live points, always above the water
controls_right(plot_flood, iteration=IntSlider(value=70, min=0, max=160, step=10, continuous_update=False, description='iteration'))

* nested sampling's job isn't to maximize. That just happens incidentally if you use it properly
* here each new point is drawn exactly from above the water
* in many dimensions codes can't do that, and can miss a narrow peak that no live point reaches. More live points lower that risk
* you decide when to stop, as with how many steps to run a chain

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**So yes, you found the maximum, but more importantly, you flooded the entire valley. You've sampled everything.**

</div>

## In-class exercise

`lecture/07/inclass_exercise_day14.ipynb`: 100 points from a straight line, unit errors. Pairs, 10 minutes, Copilot allowed.

* $M_0$, a line, $y = m\,t + c$; $M_1$, a quadratic, $y = b\,t^2 + m\,t + c$, with a flat prior $b \in [-0.1, 0.1]$
* Q1: compute $P(D \mid M)$ for both on a grid, with flat priors. Check that each prior integrates to 1
* Q2: compute $2\ln B_{1,0}$ and $\textrm{BIC}_0 - \textrm{BIC}_1$. On Day 8's scale (0-2 weak, 2-6 positive, 6-10 strong), how strong is each?
* Q3: defend your answer in two lines

<div style="font-size:0.8em; color:#666;">The setup follows the LSST-DA Data Science Fellowship Program, Session 23 (2025).</div>

In [ ]:
# The exercise's data, as the notebook draws it, with the best line and quadratic when asked (a "skip" cell)
def plot_exercise_data(fits=False):
    fig, ax = plt.subplots(figsize=(9, 3.0))
    ax.errorbar(t_ex, y_ex, yerr=sig_ex, fmt='o', ms=3, color='0.5', elinewidth=0.6)
    if fits:
        tt = np.linspace(0, 10, 200)
        p1 = np.polyfit(t_ex, y_ex, 1); p2 = np.polyfit(t_ex, y_ex, 2)
        ax.plot(tt, np.polyval(p1, tt), 'C0', lw=2.4, label='best line ($M_0$)')
        ax.plot(tt, np.polyval(p2, tt), 'C3', lw=2.0, ls='--', label='best quadratic ($M_1$)')
        ax.legend(frameon=False, loc='upper left')
    ax.set_xlabel('$t$'); ax.set_ylabel('$y$')
    plt.tight_layout(); plt.show()

In [ ]:
# the 100 points you will fit, unit error bars
plot_exercise_data()

In [ ]:
# The exercise's answers, for the debrief (a "skip" cell): grid evidences for the line and the quadratic, and BIC
_m = np.linspace(-0.5, 1.5, 101); _c = np.linspace(-0.7, 1.0, 101); _b = np.linspace(-0.1, 0.1, 61)
def _lnZ(lnL, axes, vol):
    mx = lnL.max(); w = np.exp(lnL - mx)
    for ax, g in sorted(axes, reverse=True):
        w = np.trapezoid(w, g, axis=ax)
    return np.log(w) + mx - np.log(vol)
_lline = np.sum(stats.norm.logpdf(y_ex, _m[:, None, None] * t_ex + _c[None, :, None], sig_ex), axis=-1)
_lquad = np.array([np.sum(stats.norm.logpdf(y_ex, bb * t_ex**2 + _m[:, None, None] * t_ex + _c[None, :, None], sig_ex), axis=-1) for bb in _b])
ex_lnZ0 = _lnZ(_lline, [(0, _m), (1, _c)], 2.0 * 1.7)
ex_lnZ1 = _lnZ(_lquad, [(0, _b), (1, _m), (2, _c)], 2.0 * 1.7 * 0.2)
_chi2 = lambda A: np.sum(((y_ex - A @ np.linalg.lstsq(A, y_ex, rcond=None)[0]) / sig_ex)**2)
_A0 = np.vstack([t_ex, np.ones(t_ex.size)]).T; _A1 = np.vstack([t_ex**2, t_ex, np.ones(t_ex.size)]).T
ex_dBIC = (_chi2(_A0) + 2 * np.log(t_ex.size)) - (_chi2(_A1) + 3 * np.log(t_ex.size))
ex_b_hat, ex_b_err = np.linalg.lstsq(_A1, y_ex, rcond=None)[0][0], np.sqrt(np.linalg.inv(_A1.T @ _A1)[0, 0]) * sig_ex
def exercise_numbers():
    print(f"ln P(D|line) = {ex_lnZ0:.2f},  ln P(D|quad) = {ex_lnZ1:.2f}")
    print(f"ln B_10 = {ex_lnZ1 - ex_lnZ0:.2f},  2 ln B_10 = {2 * (ex_lnZ1 - ex_lnZ0):.2f};  BIC_0 - BIC_1 = {ex_dBIC:.2f}")
    print(f"best-fit quadratic term b = {ex_b_hat:.4f} +/- {ex_b_err:.4f}, inside its prior [-0.1, 0.1]")

## Exercise results

In [ ]:
# the best line and the best quadratic, on top of each other; then the grid evidences, the Bayes factor, and BIC
plot_exercise_data(fits=True)
exercise_numbers()

* you can't tell the two fits apart by eye
* $2\ln B_{1,0} = -3.5$: "positive" (the 2-6 rung) for the line, $M_0$ - the sign says which model. That is a factor of $e^{1.7} \approx 6$
* BIC agrees on the sign, and roughly on the size
* the data pin $b$ to $\pm 0.013$, far inside the prior's width of 0.2 - the quadratic is penalized for prior volume it didn't need
* widen $b$'s prior 10$\times$ and $2\ln B$ drops by $2\ln 10 = 4.6$. The fit is the same; the prior-volume (Occam) penalty carries the change
* with seven parameters, as in Lab 05, you'd run a sampler. It gives you the posterior (for the evidence, use nested sampling)
* how do you know a sampler's answer is right?

**The evidence prefers the true model, the line - but only 6 to 1.**

## This week

* the **midterm** is posted after class today, due **Wed Oct 14 by Noon**, graded like a lab. Its Part 2 uses today's checks 2 and 3
* **Lab 05** is due **Fri Oct 9 by Noon**. After today, look again at:
    * Part 2's $\tau$: your convergence check for `emcee`
    * Part 3's `ai_solution.ipynb`: walkers started "in a tight ball around the MAP estimate", and "trace plots already confirm convergence". Today says why neither is enough
* **Quiz 3**: probably Tue Oct 20, on Bayes and MCMC (Days 12-14)
* **next week, Oct 13 and 15, I teach remotely over Zoom**, at the usual link, <a href="https://go.illinois.edu/gsn" target="_blank">go.illinois.edu/gsn</a>. Join from wherever you like
* Abha's office hours: Fridays, 1-2 pm, over Zoom
* a correction from Tuesday: the review test's generative slope was one error bar from the truth ($z = -1.0$), not two
* every date is in `ASSIGNMENTS.md`

**The midterm and Lab 05 both need today's checks: overdispersed starts, $\hat R$, ESS.**

## Detailed balance

* check 1 is a guarantee of the algorithm: a Metropolis chain visits each $\theta$ in proportion to the posterior, $p = P(\theta \mid D)$. Why?
* Day 12's rule: propose a move, and accept it with probability $\min[1, R]$, the smaller of 1 and $R = p(\theta')/p(\theta)$. If it is rejected, record $\theta$ again
* if the number of moves from $\theta$ to $\theta'$ equals the number of moves back, for every pair of states, the chain is in **detailed balance**
* if this holds, a chain that is sampling $p$ keeps sampling $p$ - $p$ is the chain's **stationary** distribution

In [ ]:
# Detailed balance with two states: 10 copies of a chain, and the flows between the states (a "skip" cell)
from matplotlib.patches import FancyBboxPatch
def plot_rooms():
    fig, ax = plt.subplots(figsize=(10, 3.1))
    for x0, n, lab in [(0.0, 2, r'$\theta_1$:  $p = 0.2$'), (7.0, 8, r'$\theta_2$:  $p = 0.8$')]:
        ax.add_patch(FancyBboxPatch((x0, 0), 3.3, 2.4, boxstyle='round,pad=0.05', fc='0.95', ec='0.3', lw=1.5))
        ax.text(x0 + 1.65, 2.65, lab, ha='center', fontsize=14)
        for k in range(n):
            ax.plot(x0 + 0.45 + (k % 4) * 0.8, 1.75 - (k // 4) * 0.9, 'o', color='C0', ms=17)
    ax.annotate('', xy=(6.85, 1.6), xytext=(3.45, 1.6), arrowprops=dict(arrowstyle='->', color='C0', lw=3))
    ax.text(5.15, 1.75, 'to higher p: always accepted\n2 x 1 = 2 move', ha='center', va='bottom', color='C0', fontsize=12)
    ax.annotate('', xy=(3.45, 0.8), xytext=(6.85, 0.8), arrowprops=dict(arrowstyle='->', color='C3', lw=3))
    ax.text(5.15, 0.65, 'to lower p: accepted\n1/4 of the time\n8 x 1/4 = 2 move', ha='center', va='top', color='C3', fontsize=12)
    ax.set_xlim(-0.3, 10.7); ax.set_ylim(-0.6, 3.1); ax.axis('off')
    plt.tight_layout(); plt.show()

In [ ]:
# 10 copies of one chain: 2 at the low state, 8 at the high one, and the flows each step
plot_rooms()

* 10 copies of one chain: 2 at $\theta_1$ ($p = 0.2$), 8 at $\theta_2$ ($p = 0.8$)
* start at 5 and 5 instead: the counts settle at 2 and 8
* one long chain behaves like the 10 copies
* from $\theta_2$, a move down is refused 3/4 of the time - the chain stays put and records $\theta_2$ again
* <a href="https://setosa.io/markov/index.html#%7B%22tm%22%3A%5B%5B0%2C1%5D%2C%5B0.25%2C0.75%5D%5D%7D" target="_blank">launch the two-state chain</a> (A = $\theta_1$, B = $\theta_2$). Watch the dot: it sits at B about four times as long as at A

**Equal flows each way keep a Metropolis chain on the posterior.**

## A three-state chain

* three values of $\theta$, with $p = 0.2$, 0.5 and 0.3. Propose either other state, accept with $\min[1, R]$
* in the plot: blue bars = the fraction of steps in each state; red lines = $p$
* slide the steps to 10, 1000, then 100,000. Watch: the bars move onto the red lines
* then untick "keep rejected steps". Watch: the bars move off them

In [ ]:
# A 3-state Metropolis chain: occupation fractions against p, with and without the rejected steps (a "skip" cell)
P3 = np.array([0.2, 0.5, 0.3])
def _run3(n=100_000, seed=8):
    rng = np.random.default_rng(seed)
    s, states, moved = 0, np.empty(n, int), np.zeros(n, bool)
    for k in range(n):
        sp = (s + rng.integers(1, 3)) % 3                         # one of the other two states, equally likely
        if rng.uniform() < min(1.0, P3[sp] / P3[s]):
            s, moved[k] = sp, True
        states[k] = s
    return states, moved
chain3, moved3 = _run3()
def plot_three(steps=100, keep_rejected=True):
    st = chain3[:steps] if keep_rejected else chain3[:steps][moved3[:steps]]
    frac = np.bincount(st, minlength=3) / max(st.size, 1)
    fig, ax = plt.subplots(figsize=(6.8, 3.5))
    ax.bar([0, 1, 2], frac, width=0.55, color='C0', label='fraction of steps in the chain')
    ax.hlines(P3, np.arange(3) - 0.33, np.arange(3) + 0.33, color='C3', lw=3, label='the posterior $p$')
    ax.set_xticks([0, 1, 2]); ax.set_xticklabels([r'$\theta_1$', r'$\theta_2$', r'$\theta_3$']); ax.set_ylim(0, 0.75)
    ax.set_title(f"{steps} steps, {'all kept' if keep_rejected else 'rejected steps dropped'}: " + ', '.join(f'{f:.3f}' for f in frac), fontsize=12)
    ax.legend(frameon=False, loc='upper left', fontsize=11)
    plt.tight_layout(); plt.show()

In [ ]:
# blue bars: the fraction of steps in each state; red: p
controls_right(plot_three, steps=SelectionSlider(options=[10, 100, 1000, 10_000, 100_000], value=100, continuous_update=False, description='steps'),
               keep_rejected=Checkbox(value=True, description='keep rejected steps'))

* by 100,000 steps the fractions match $p$ to within 0.001
* drop the rejected steps and the answer is wrong, about $0.29, 0.36, 0.36$
* the peak, $\theta_2$, is undercounted: it is where the most proposals get rejected

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Even if a step is rejected, we still keep a sample. The difficulty of finding a temptingly better point is important information!**

</div>

<div style="font-size:0.8em; color:#666;">Wording after A. Mantz, <i>StatisticalMethods</i> lessons.</div>

## MCMC diagnostics

* check 2: what would make us confident of convergence?
* each chain is stationary - it has stopped drifting
* chains from overdispersed starts - spread wider than the posterior - end up in the same place. Then they are **well mixed**
* a chain gets there eventually, but nothing says when
* four made-up runs, three chains each. Which would you trust: A, B, C or D?

In [ ]:
# Four made-up sets of three chains, to judge by eye (a "skip" cell)
def _poll_chains(seed=5, n=500):
    rng = np.random.default_rng(seed); out = {}
    starts = np.array([-3.0, 0.5, 3.5])
    def ar1(mu, phi, sd, x0):
        x = np.empty(n); x[0] = x0
        for i in range(1, n):
            x[i] = mu + phi * (x[i - 1] - mu) + sd * rng.normal()
        return x
    out['A'] = np.array([ar1(0.0, 0.8, 0.6, s) for s in starts])                          # mixed: all three look alike
    out['B'] = np.array([np.cumsum(np.r_[s, rng.normal(0, 0.5, n - 1)]) for s in starts])   # random walks: never settle
    out['C'] = np.array([ar1(m, 0.8, 0.5, s) for m, s in zip([-2.5, -2.5, 2.5], starts)])   # each settles, not all in one place
    out['D'] = np.array([ar1(0.0, 0.998, 0.05, s) for s in 2 * starts])                   # sticky: still drifting toward each other
    return out
poll = _poll_chains()
def plot_poll():
    fig, axs = plt.subplots(2, 2, figsize=(11, 4.6), sharex=True)
    for ax, lab in zip(axs.flat, 'ABCD'):
        for j, col in enumerate(['C0', 'C3', 'C2']):
            ax.plot(poll[lab][j], color=col, lw=0.8)
        ax.text(0.02, 0.85, lab, transform=ax.transAxes, fontsize=16, weight='bold')
        ax.set_yticks([])
    for ax in axs[1]:
        ax.set_xlabel('step')
    plt.tight_layout(); plt.show()

In [ ]:
# A-D: three chains each, from different starting points
plot_poll()

* A is stationary and well mixed: all three look the same
* B is non-stationary: random walks that never settle
* C is not mixed: each chain is stationary, but one is somewhere else
* D has not converged yet: the chains are still drifting towards each other

**Only A passes - only comparing chains catches C.**

## Gelman-Rubin $\hat R$

* treat each chain as a data set, as on Day 5: if all the chains sample the same $p$, their means agree to within $\sigma/\sqrt{N_{\rm eff}}$
    * $N_{\rm eff} = N/\tau$, Day 12's effective number of samples (`ArviZ` calls it the ESS)
* for a given parameter, $\theta$, the $\hat R$ statistic compares the variance across chains with the variance within a chain
* intuitively, if the chains are sitting in very different places, i.e. not sampling the same $p$, $\hat R$ will be large
* we'd like to see $\hat R \approx 1$. `az.rhat`, from the `ArviZ` package, computes it
* $\hat R$ is not the Metropolis ratio $R$

<div style="font-size:0.8em; color:#666;">Gelman &amp; Rubin 1992, Statist. Sci. 7, 457. Wording after A. Mantz, <i>StatisticalMethods</i> lessons.</div>

In [ ]:
# The idea behind R-hat on runs A and C from the poll: each chain's mean with its error bar, and the two spreads (a "skip" cell)
def plot_gr_idea():
    fig, axs = plt.subplots(1, 2, figsize=(12, 3.3), sharey=True)
    for ax, lab in zip(axs, 'AC'):
        ch = poll[lab][:, 100:]                                  # first 100 steps dropped as burn-in
        bins = np.linspace(-5, 5, 61)
        ax.hist(ch.ravel(), bins=bins, color='0.85', density=True, label='all chains pooled')
        for j, col in enumerate(['C0', 'C3', 'C2']):
            ax.hist(ch[j], bins=bins, histtype='step', color=col, lw=1.6, density=True)
            err = ch[j].std(ddof=1) / np.sqrt(float(az.ess(ch[j][None, :])))
            ax.errorbar(ch[j].mean(), 0.78 + 0.07 * j, xerr=err, fmt='o', color=col, ms=6, capsize=3)
        within, pooled = np.mean(ch.var(axis=1, ddof=1)), ch.var(ddof=1)
        ax.set_title(f'{lab}: pooled variance {pooled:.2f}, within one chain {within:.2f};  ' + r'$\hat R \approx$' + f' {np.sqrt(pooled / within):.2f}', fontsize=12)
        ax.set_xlabel(r'$\theta$'); ax.set_yticks([]); ax.set_ylim(0, 1.0)
    axs[0].text(-4.8, 0.85, 'chain means,\n' + r'$\pm\sigma/\sqrt{N_{\rm eff}}$', fontsize=11, va='center')
    plt.tight_layout(); plt.show()

In [ ]:
# runs A and C again: lines, each chain's samples; grey, all pooled; dots, each chain's mean and its error bar
plot_gr_idea()

* A: the three means overlap within their error bars, and the pooled and within-chain variances are about equal: $\hat R \approx 1$
* C: the green chain's mean sits about 30 error bars from the other two, and the pooled variance is 9 times the within-chain variance

**$\hat R$ asks: do the chain means disagree by more than their error bars?**

## RX J1131-1231, from Day 12

<div style="display:flex; align-items:flex-start; gap:1.2em;">
<div>

* one quasar, lensed into four images by galaxy G. Image D shows the quasar $\Delta t$ days after image A
* on Day 12 we sampled the delay $\Delta t$, and an extra noise term (the jitter), with Metropolis: $\Delta t = 94.6 \pm 1.3$ d
* the posterior has a second peak near 215 d, from the gaps between observing seasons. A chain started at 215 d got stuck there

</div>
<div style="flex:0 0 26%; text-align:center;">

<img src="images/rxj1131_hubble_heic1702d_labelled.png" alt="Hubble image of RX J1131-1231: the lens galaxy G at the centre, three bright quasar images B, A and C on an arc to the left, and a fainter fourth image D to the right" style="max-height:200px">

<div style="font-size:0.7em; color:#666;">ESA/Hubble, NASA, Suyu et al. (heic1702d, CC BY 4.0); labels added.</div>

</div>
</div>

In [ ]:
# Day 12's RX J1131-1231 time-delay posterior (copied from the Day 12 deck), and four Metropolis chains (a "skip" cell)
def load_csv(path):
    with open(path) as _f:
        _nhead = sum(1 for _line in _f if _line.startswith('#'))
    return np.genfromtxt(path, delimiter=',', skip_header=_nhead, names=True, dtype=None, encoding='utf-8')
lc = load_csv('../06/data/rxj1131_cosmograil_millon2020.csv')
recent = lc['hjd'] > 56200                       # the last six observing seasons, Dec 2012 to 2018
t_lc = lc['hjd'][recent]; mA, eA = lc['mag_A'][recent], lc['err_A'][recent]; mD, eD = lc['mag_D'][recent], lc['err_D'][recent]
N_LC = t_lc.size
M0 = np.median(mA)
T0, T1 = t_lc.min() - 250, t_lc.max() + 50
KN = np.r_[[T0] * 3, np.arange(T0, T1, 20.0), [T1] * 4]                                    # quasar curve q: a knot every 20 d
KM = np.r_[[t_lc.min() - 1] * 3, np.arange(t_lc.min() - 1, t_lc.max() + 1, 300.0), [t_lc.max() + 1] * 4]   # microlensing m: every 300 d
B_A = BSpline.design_matrix(t_lc, KN, 3).toarray(); B_ML = BSpline.design_matrix(t_lc, KM, 3).toarray()
N_I, N_M = B_A.shape[1], B_ML.shape[1]
SIG_W = 1.0
def log_post_lens(theta):
    dt, ln_s = theta
    if not (0 < dt < 400 and -8 < ln_s < -1):
        return -np.inf
    s = np.exp(ln_s)
    B_D = BSpline.design_matrix(np.clip(t_lc - dt, T0, T1 - 1e-6), KN, 3).toarray()
    wA, wD = 1 / np.sqrt(eA**2 + s**2), 1 / np.sqrt(eD**2 + s**2)
    X = np.zeros((2 * N_LC, N_I + N_M)); X[:N_LC, :N_I] = B_A * wA[:, None]
    X[N_LC:, :N_I] = B_D * wD[:, None]; X[N_LC:, N_I:] = B_ML * wD[:, None]
    y = np.r_[(mA - M0) * wA, (mD - M0) * wD]
    Aprec = X.T @ X + np.eye(N_I + N_M) / SIG_W**2
    c = np.linalg.solve(Aprec, X.T @ y); r = y - X @ c
    with np.errstate(all='ignore'):                  # Apple's Accelerate BLAS (pulled in with PyMC) raises spurious warnings here; the value is right
        logdet = np.linalg.slogdet(Aprec)[1]
    return (-0.5 * (r @ r + c @ c / SIG_W**2) + np.sum(np.log(wA)) + np.sum(np.log(wD))
            - 0.5 * logdet - (N_I + N_M) * np.log(SIG_W))
def metropolis(log_post, theta0, step, n_steps, seed=1):
    # Day 12's sampler, unchanged
    rng = np.random.default_rng(seed)
    theta = np.array(theta0, dtype=float); logp = log_post(theta)
    chain = np.zeros((n_steps, theta.size)); n_accept = 0
    for i in range(n_steps):
        proposal = theta + rng.normal(0, step, size=theta.size)
        logp_new = log_post(proposal)
        if np.log(rng.uniform()) < logp_new - logp:
            theta, logp = proposal, logp_new; n_accept += 1
        chain[i] = theta
    return chain, n_accept / n_steps
STEP_LENS = np.array([1.547, 0.082])                 # Day 12's step: 1.2 x each parameter's curvature error bar
RH_FIXED = [70, 110, 150]; RH_STARTS = [130, 170, 190, 200, 230]
def _run_rhat():
    out = {}
    for j, st in enumerate(RH_FIXED + RH_STARTS):
        out[f'chain_{st}'] = metropolis(log_post_lens, [float(st), -3.0], STEP_LENS, 3000, seed=100 + j)[0][:, 0]
    return out
rh = cached('rhat', _run_rhat)
RH_BURN = 300
def gelman_rubin(chains):
    # chains: an (m, n) array, burn-in already removed. B, W, V and R-hat as on the EXTRA Gelman-Rubin slide
    m, n = chains.shape
    B = n / (m - 1) * np.sum((chains.mean(axis=1) - chains.mean())**2)
    W = np.mean(chains.var(axis=1, ddof=1))
    V = (n - 1) / n * W + B / n
    return np.sqrt(V / W)
RH_COLS = ['C0', 'C2', 'C1', 'C3']
def plot_rhat(start4=170):
    ch = np.array([rh[f'chain_{s}'] for s in RH_FIXED + [start4]])
    fig, (ax, az_) = plt.subplots(1, 2, figsize=(12, 3.7), gridspec_kw=dict(width_ratios=[2.2, 1]))
    for j, col in enumerate(RH_COLS):
        ax.plot(ch[j], color=col, lw=0.7, label=f'chain {j + 1}: start {[*RH_FIXED, start4][j]} d')
    ax.axvspan(0, RH_BURN, color='0.85', zorder=0)
    ax.set_xlabel('step'); ax.set_ylabel(r'delay $\Delta t$ [d]'); ax.set_ylim(55, 250)
    leg = ax.legend(frameon=False, fontsize=11, ncol=4, loc='lower center', bbox_to_anchor=(0.5, 1.0), handlelength=1.5)
    for h in leg.get_lines():
        h.set_linewidth(3)
    post = ch[:, RH_BURN:]
    for j, col in enumerate(RH_COLS):
        az_.hist(post[j], bins=np.arange(85, 230, 1.5), color=col, histtype='step', lw=1.8, orientation='horizontal')
    az_.set_ylim(55, 250); az_.set_xlabel('samples'); az_.set_yticks([])
    r = gelman_rubin(post)
    az_.set_title(rf'$\hat R$ = {r:.3f}: ' + ('well mixed' if r < 1.01 else 'NOT mixed'), fontsize=13)
    plt.tight_layout(); plt.show()
    print(f"R-hat by hand = {r:.3f};  ArviZ's rank-normalized split R-hat = {float(az.rhat(post)):.3f};  pooled median delay {np.median(post):.1f} d")
def plot_lc():
    # the last three seasons: image D shifted back by 94.6 d, and by its median offset, follows image A
    # (the slow offsets left between them are the microlensing that Day 12's model fit)
    wa, wd = t_lc > 57300, t_lc - 94.6 > 57300
    fig, ax = plt.subplots(figsize=(11, 2.7))
    ax.errorbar(t_lc[wa], mA[wa], eA[wa], fmt='o', ms=2.5, color='C0', elinewidth=0.5, label='image A')
    ax.errorbar(t_lc[wd] - 94.6, mD[wd] - np.median(mD[wd]) + np.median(mA[wa]), eD[wd], fmt='o', ms=2.5, color='C3', elinewidth=0.5,
                label='image D, shifted 94.6 d earlier')
    ax.invert_yaxis(); ax.set_xlabel('HJD - 2,400,000 [d]'); ax.set_ylabel('magnitude (relative)')
    ax.legend(frameon=False, ncol=2, loc='lower center', bbox_to_anchor=(0.5, 1.0))
    plt.tight_layout(); plt.show()

In [ ]:
# Day 12's light curves: image D, shifted back by the delay, follows image A; the slow offsets are microlensing
plot_lc()

**The delay is about 95 days; a chain can get stuck at 215.**

## Gelman-Rubin on RX J1131

* I've started four Metropolis chains for the delay, from 70, 110 and 150 d and one you pick. The first 300 steps are dropped as burn-in
* slide chain 4's start to 200, then back to 170. Watch $\hat R$ in the right panel's title: about 1 when all four chains overlap, large when chain 4 sits at 215 d

In [ ]:
# four chains for the delay (grey: burn-in, dropped); right: each chain's samples, and R-hat by hand
controls_right(plot_rhat, start4=SelectionSlider(options=RH_STARTS, value=170, continuous_update=False, description='start 4 [d]'))

* for starts up to 190 d, $\hat R \approx 1.00$ and all four chains agree. From 200 d, $\hat R = 35$: a quarter of the samples sit at the wrong delay
* `az.rhat` computes $\hat R$ a little differently (EXTRA slide) and gives 1.54 here. Both are far above `ArviZ`'s threshold of 1.01
* `emcee`'s walkers are not *independent* chains! For $\hat R$ with `emcee`, run separate ensembles from separate starts and treat each as one chain (midterm Part 2)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**In general, the only way to discover that other peaks exist is by exploring the parameter space with many widely dispersed chains.**

</div>

<div style="font-size:0.8em; color:#666;">Wording after A. Mantz, <i>StatisticalMethods</i> lessons.</div>

## Hamiltonian Monte Carlo

<div style="display:flex; align-items:flex-start; gap:1.2em;">
<div>

* check 3: are there enough independent samples? $N_{\rm eff} = N/\tau$ (Day 12), which `ArviZ` calls the ESS
* Metropolis meanders to and fro, so successive samples are correlated and the ESS is small
* so far our samplers need only likelihoods and priors, NOT THEIR DERIVATIVES. HMC uses the gradient of $\ln p$
* turn $\ln p$ upside down into a bowl, $-\ln p$. Put a ball in it, give it a random kick, and let it roll - where it ends up is the next proposal

**HMC uses the gradient to make long moves along the posterior.**

</div>
<div style="flex:0 0 40%; text-align:center;">

<img src="images/dong2022_dsfp_hmc_bowl.png" alt="A 3-D bowl-shaped surface of minus the log posterior over two parameters m and b, with a dotted path curving down and around the bowl from a black starting dot" style="max-height:300px">

<div style="font-size:0.7em; color:#666;">J. Dong, LSST-DA Data Science Fellowship Program, Session 16 (2022), MCMC lecture, p. 18.</div>

</div>
</div>

In [ ]:
# A ball rolling in the bowl -ln p, by leapfrog or by Euler steps (a "skip" cell). 1-D Gaussian posterior, width 1
def roll(theta0, v0, eps, L, euler=False):
    grad = lambda th: -th                                     # d ln p / d theta: the force on the ball
    th, v = [theta0], [v0]
    for _ in range(L):
        if euler:                                             # both updates use the old values
            th.append(th[-1] + eps * v[-1]); v.append(v[-1] + eps * grad(th[-2]))
        else:                                                 # leapfrog: half kick, drift, half kick
            vh = v[-1] + 0.5 * eps * grad(th[-1]); th.append(th[-1] + eps * vh); v.append(vh + 0.5 * eps * grad(th[-1]))
    th, v = np.array(th), np.array(v)
    return th, v, 0.5 * th**2 + 0.5 * v**2                    # H = -ln p + v^2/2
BOWL = np.linspace(-3.2, 3.2, 300)

import os
from matplotlib.animation import FuncAnimation, PillowWriter
LEAP_GIF = 'images/day14_leapfrog_vs_euler.gif'
def make_leapfrog_gif(path=LEAP_GIF, eps=0.3, L=30):
    # leapfrog (blue) and Euler (red) from the same start and kick; left: the bowl, right: the energy H
    thl, vl, Hl = roll(-2.2, 0.8, eps, L); the, ve, He = roll(-2.2, 0.8, eps, L, euler=True)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
    a1.plot(BOWL, 0.5 * BOWL**2, 'k', lw=2); a1.set_xlim(-3.4, 3.4); a1.set_ylim(-0.2, 5.6)
    a1.set_xlabel(r'$\theta$'); a1.set_ylabel(r'$-\ln p(\theta)$')
    bl, = a1.plot([], [], 'o', color='C0', ms=14, label='leapfrog'); be, = a1.plot([], [], 'o', color='C3', ms=14, label='Euler')
    a1.legend(frameon=False, loc='upper center', ncol=2)
    ll, = a2.plot([], [], 'C0', lw=2.5, label='leapfrog'); le, = a2.plot([], [], 'C3', lw=2.5, label='Euler')
    a2.set_xlim(0, L); a2.set_ylim(0, 10); a2.set_xlabel('step'); a2.set_ylabel('energy $H$'); a2.legend(frameon=False, loc='upper left')
    note = a2.text(0.98, 0.9, '', transform=a2.transAxes, ha='right', color='C3', fontsize=12)
    def draw(k):
        bl.set_data([thl[k]], [0.5 * thl[k]**2]); be.set_data([the[k]], [0.5 * the[k]**2])
        ll.set_data(np.arange(k + 1), Hl[:k + 1]); le.set_data(np.arange(k + 1), He[:k + 1])
        a1.set_title(f'step {k} of {L}', fontsize=12)
        note.set_text('Euler leaves the bowl' if abs(the[k]) > 3.4 else '')
        return bl, be, ll, le
    anim = FuncAnimation(fig, draw, frames=L + 1)
    plt.tight_layout(); anim.save(path, writer=PillowWriter(fps=3)); plt.close(fig)
if RECOMPUTE or not os.path.exists(LEAP_GIF):
    make_leapfrog_gif()

## One HMC iteration

* give the ball a random kick: a velocity $v$
* roll it for $L$ small time steps of size $\epsilon$ (here $L$ counts steps; it is not the likelihood)
* physical interpretation:
    * $\theta$ = position, $v$ = velocity
    * $-\ln p(\theta)$ = potential energy, $\tfrac12 v^2$ = kinetic energy
* $H$ = total energy = potential + kinetic, the **Hamiltonian**. By conservation of energy, $H$ remains constant as the ball rolls
* at the end, accept the new point with probability $\min[1, e^{-\Delta H}]$ - Day 12's rule with $R = e^{-\Delta H}$. If $H$ barely changes, the point is almost always accepted

**Energy conservation is what lets HMC accept long jumps.**

## Euler vs. leapfrog

* a computer rolls the ball in steps, and careless steps add energy
* **Euler's method** updates the position and the velocity together, both from their old values. The energy grows by about 9% every step, and the growth compounds
* **leapfrog** first changes the velocity by half a step. Then it moves the position a full step. Then it changes the velocity by the other half. $H$ stays almost exactly constant
* in the animation, both balls start from the same place with the same kick. Watch: the red (Euler) energy line climbs off the plot while the blue one stays flat

<img src="images/day14_leapfrog_vs_euler.gif" alt="Animation: two balls start at the same point in a bowl-shaped curve. The blue leapfrog ball rolls back and forth and its energy line stays flat; the red Euler ball swings higher each pass and leaves the bowl while its energy line climbs off the plot" style="max-height:290px; display:block; margin:0 auto;">

**Leapfrog keeps $H$ constant; Euler's ball flies out of the bowl.**

## HMC vs. Metropolis

* the posterior is a long, narrow ridge in $p$ - a valley in $-\ln p$: a 2-D Gaussian with correlation 0.95, like two strongly correlated parameters
* left: HMC with $L = 10$ leapfrog steps per iteration (faint: the rolled paths). Right: Metropolis with 10 chain steps per iteration - the same cost
* watch the red points, one per iteration: HMC's take big jumps along the ridge, while Metropolis's inch along it

In [ ]:
# Hand-written HMC on a correlated 2-D Gaussian, against random-walk Metropolis (a "skip" cell)
RHO = 0.95
CINV = np.linalg.inv(np.array([[1, RHO], [RHO, 1]]))
logp2 = lambda x: -0.5 * x @ CINV @ x
grad2 = lambda x: -CINV @ x                                    # the gradient of ln p: the force on the ball
def hmc(logp, grad, x0, eps, L, n_iter, seed=0, keep_path=0):
    rng = np.random.default_rng(seed)
    x = np.array(x0, float); chain = np.empty((n_iter, x.size)); paths = []; n_acc = 0
    for k in range(n_iter):
        v = rng.normal(size=x.size)                            # fresh momentum, unit masses
        xn, vn = x.copy(), v + 0.5 * eps * grad(x)             # leapfrog: half kick,
        path = [xn.copy()]
        for i in range(L):
            xn = xn + eps * vn                                 # drift,
            if i < L - 1:
                vn = vn + eps * grad(xn)                       # full kick
            path.append(xn.copy())
        vn = vn + 0.5 * eps * grad(xn)                         # final half kick
        dH = (-logp(xn) + 0.5 * vn @ vn) - (-logp(x) + 0.5 * v @ v)
        if np.log(rng.uniform()) < -dH:                        # accept with probability min[1, exp(-Delta H)]
            x = xn; n_acc += 1
        chain[k] = x
        if k < keep_path:
            paths.append(np.array(path))
    return chain, n_acc / n_iter, paths
EPS2, HMC_L = 0.18, [1, 5, 10, 25, 35, 50]
def _run_hmc2():
    out = {}
    for L in HMC_L:
        ch, acc, paths = hmc(logp2, grad2, [-2.5, -2.0], EPS2, L, 3000, seed=1, keep_path=15)
        out[f'chain_{L}'], out[f'acc_{L}'] = ch, acc
        out[f'paths_{L}'] = np.concatenate([np.c_[p, np.full(len(p), i)] for i, p in enumerate(paths)])
        rw, racc = metropolis(logp2, [-2.5, -2.0], np.full(2, 0.35), 3000 * L, seed=2)
        out[f'rw_{L}'], out[f'rwacc_{L}'] = rw[::L], racc                 # count L Metropolis steps as one iteration, as Neal (2011) does
    return out
h2 = cached('hmc2', _run_hmc2)
def plot_hmc(L=10):
    ch, acc, pth, rw = h2[f'chain_{L}'], float(h2[f'acc_{L}']), h2[f'paths_{L}'], h2[f'rw_{L}']
    g = np.linspace(-3.5, 3.5, 200); X, Y = np.meshgrid(g, g); Z = np.exp(-0.5 * (CINV[0, 0] * X**2 + 2 * CINV[0, 1] * X * Y + CINV[1, 1] * Y**2))
    fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(13, 3.9), gridspec_kw=dict(width_ratios=[1, 1, 1.5]))
    for a, title in [(a1, f'HMC, L = {L}: acceptance {acc:.0%}'), (a2, f"Metropolis, {L} step{'s' if L > 1 else ''} per iteration")]:
        a.contour(X, Y, Z, levels=[0.05, 0.3, 0.7], colors='0.6', linewidths=1); a.set_aspect('equal')
        a.set_xlim(-3.5, 3.5); a.set_ylim(-3.5, 3.5); a.set_title(title, fontsize=12); a.set_xlabel(r'$\theta_1$')
    for i in range(15):
        p = pth[pth[:, 2] == i]; a1.plot(p[:, 0], p[:, 1], color='C0', lw=0.6, alpha=0.5)
    a1.plot(*np.r_[[[-2.5, -2.0]], ch[:15]].T, 'o-', color='C3', ms=4, lw=1.0); a1.set_ylabel(r'$\theta_2$')
    a2.plot(*np.r_[[[-2.5, -2.0]], rw[:15]].T, 'o-', color='C3', ms=4, lw=1.0)
    e_h, e_r = ess_per_iter(ch[200:, 0]), ess_per_iter(rw[200:, 0])
    a3.plot(rw[:150, 0], color='0.55', lw=1.0)
    a3.plot(ch[:150, 0], color='C0', lw=1.3)
    a3.set_title(f'ESS per iteration: HMC {e_h:.2g} (blue), Metropolis {e_r:.2g}', fontsize=11)
    a3.set_xlabel('iteration'); a3.set_ylabel(r'$\theta_1$')
    plt.tight_layout(); plt.show()
HMC_GIF = 'images/day14_hmc_vs_metropolis.gif'
def make_hmc_gif(path=HMC_GIF, L=10, n=20):
    # the first n iterations of HMC (L leapfrog steps each) and of Metropolis (L steps each), at equal cost
    ch, _, paths = hmc(logp2, grad2, [-2.5, -2.0], EPS2, L, n, seed=1, keep_path=n)
    rw = metropolis(logp2, [-2.5, -2.0], np.full(2, 0.35), n * L, seed=2)[0]
    g = np.linspace(-3.5, 3.5, 200); X, Y = np.meshgrid(g, g); Z = np.exp(-0.5 * (CINV[0, 0] * X**2 + 2 * CINV[0, 1] * X * Y + CINV[1, 1] * Y**2))
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4.4))
    for a in (a1, a2):
        a.contour(X, Y, Z, levels=[0.05, 0.3, 0.7], colors='0.6', linewidths=1); a.set_aspect('equal')
        a.set_xlim(-3.5, 3.5); a.set_ylim(-3.5, 3.5); a.set_xlabel(r'$\theta_1$')
    a1.set_ylabel(r'$\theta_2$')
    def draw(k):
        for a in (a1, a2):
            for art in list(a.lines):
                art.remove()
        for p in paths[:k]:
            a1.plot(p[:, 0], p[:, 1], color='C0', lw=1.0, alpha=0.6)
        a1.plot(*np.r_[[[-2.5, -2.0]], ch[:k]].T, 'o-', color='C3', ms=4, lw=1.0)
        a2.plot(*np.r_[[[-2.5, -2.0]], rw[:k * L]].T, '-', color='0.6', lw=0.6)
        a2.plot(*np.r_[[[-2.5, -2.0]], rw[L - 1:k * L:L]].T, 'o-', color='C3', ms=4, lw=1.0)
        a1.set_title(f'HMC: {k} iterations', fontsize=12); a2.set_title(f'Metropolis: {k} iterations ({L} steps each)', fontsize=12)
    draw(n)                                                  # lay out with full-length titles
    plt.tight_layout()
    anim = FuncAnimation(fig, draw, frames=n + 1)
    anim.save(path, writer=PillowWriter(fps=2)); plt.close(fig)
if RECOMPUTE or not os.path.exists(HMC_GIF):
    make_hmc_gif()

<img src="images/day14_hmc_vs_metropolis.gif" alt="Animation on a long tilted ellipse: on the left, HMC points jump from one end of the ellipse to the other within a few iterations; on the right, Metropolis points creep slowly up from the lower-left end" style="max-height:300px; display:block; margin:0 auto;">

* HMC: 0.56 independent samples per iteration. Metropolis: 0.08
* in 100 dimensions the difference is far larger (Neal 2011, EXTRA slide)

**For the same cost, HMC gives about 7 times more independent samples.**

## HMC with `PyMC`

<div style="display:flex; align-items:flex-start; gap:1.2em;">
<div>

* the exercise's quadratic, same data and priors. All the code you write:

```python
import pymc as pm
with pm.Model():
    b = pm.Uniform('b', -0.1, 0.1)
    m = pm.Uniform('m', -0.5, 1.5)
    c = pm.Uniform('c', -0.7, 1.0)
    pm.Normal('y', mu=b * t**2 + m * t + c, sigma=sigma, observed=y)
    idata = pm.sample(2000, tune=1000, chains=4)
```

* `tune=1000` is burn-in: NUTS uses those steps to pick the step size $\epsilon$, then drops them
* too many leapfrog steps waste work: at $L = 50$ the ball rolls out and back - a U-turn - and HMC got only 0.006 independent samples per iteration
* NUTS, the No-U-Turn Sampler, stops each path when it starts to turn back (figure)
* install: `conda install -c conda-forge pymc arviz` (in `INSTALL.txt`). You need `ArviZ` for the midterm

</div>
<div style="flex:0 0 36%; text-align:center;">

<img src="images/hoffman2014_fig2_nuts_trajectory.png" alt="One NUTS trajectory inside a narrow tilted ellipse: open circles mark leapfrog positions zig-zagging up the ellipse from a solid black start, with arrows at the far end showing where the path turns back" style="max-height:260px">

<div style="font-size:0.7em; color:#666;">Hoffman &amp; Gelman 2014, JMLR 15, 1593 (arXiv:1111.4246), Fig. 2: one NUTS path. It stops when the velocity $v$ (magenta) turns back against the path's direction (blue).</div>

</div>
</div>

In [ ]:
# The exercise's quadratic in PyMC: uniform priors as in the exercise, NUTS, four chains (a "skip" cell).
# The run is cached (about 10 s live); the code below is exactly what produced it.
def _run_pymc():
    import pymc as pm
    with pm.Model():
        b = pm.Uniform('b', -0.1, 0.1)
        m = pm.Uniform('m', -0.5, 1.5)
        c = pm.Uniform('c', -0.7, 1.0)
        pm.Normal('y', mu=b * t_ex**2 + m * t_ex + c, sigma=sig_ex, observed=y_ex)
        idata = pm.sample(2000, tune=1000, chains=4, cores=1, random_seed=14, progressbar=False)
    return {k: idata.posterior[k].values for k in ['b', 'm', 'c']}       # each (4 chains, 2000 draws)
idata = az.from_dict({'posterior': cached('pymc', _run_pymc)})
def pymc_summary():
    return az.summary(idata)[['mean', 'sd', 'eti89_lb', 'eti89_ub', 'ess_bulk', 'ess_tail', 'r_hat']]
def plot_pymc_trace():
    fig, axs = plt.subplots(1, 3, figsize=(13, 3.0))
    for ax, name in zip(axs, ['b', 'm', 'c']):
        v = idata.posterior[name].values
        for j in range(v.shape[0]):
            ax.plot(v[j], lw=0.5, alpha=0.8)
        ax.set_title(name, fontsize=13); ax.set_xlabel('iteration')
    plt.tight_layout(); plt.show()

In [ ]:
# az.summary of the four NUTS chains, then their traces
display(pymc_summary())
plot_pymc_trace()

* the table reports checks 2 and 3: `r_hat` = 1.00 for every parameter, and `ess_bulk`, `ess_tail` are the effective samples in the middle and in the tails. Check 1 is built into NUTS
* `eti89_lb`, `eti89_ub`: `ArviZ`'s default 89% interval. For our 16th-84th: `az.summary(idata, ci_prob=0.68)`
* $b = -0.005 \pm 0.012$: the posterior includes 0. NUTS gives no evidence - for that, nested sampling
* the traces: all four chains overlap, like run A. $c$'s trace stops at $-0.7$, the edge of its prior

**You write the model; `PyMC` runs NUTS and reports $\hat R$ and ESS.**

## Wrapping up

You should be comfortable with:

* nested sampling as a rising water level, and why it can miss a narrow peak
* the exercise: the evidence prefers the line, $2\ln B_{1,0} = -3.5$
* detailed balance: equal flows between every pair of states, and why a rejected step is still a sample
* $\hat R$: run several chains from overdispersed starts and compare them. `ArviZ` wants $\hat R < 1.01$
* HMC: a ball rolled with leapfrog steps can cross the posterior in one iteration if $L$ and $\epsilon$ are right. NUTS, in `PyMC`, chooses them for you

**How do we know a sampler's answer is right? It samples $p$ (detailed balance), several chains from overdispersed starts agree ($\hat R \approx 1$), and there are enough independent samples (ESS).**

Next week, Oct 13 and 15, I'm traveling and will teach remotely over Zoom, at the usual link: <a href="https://go.illinois.edu/gsn" target="_blank">go.illinois.edu/gsn</a> - join from wherever you like. Topic: visualization as verification - plots that catch broken models before referees do

## Tue Oct 13, 3:45 pm, 134 Astronomy: Mathilde Van Cuyck (Illinois)

<div style="display:flex; align-items:flex-start; gap:1.4em; margin-top:0.4em;">
<img src="images/speaker_mathilde_van_cuyck.jpg" alt="Photo of astronomer Mathilde Van Cuyck in front of a large radio dish on a high desert plateau" style="max-height:300px; border-radius:4px;">
<div>

* **"Mapping Cosmic Star Formation in 3D with [CII] Line-Intensity Mapping"**
* [CII] is a bright infrared emission line of ionized carbon, which traces star formation
* line-intensity mapping measures the summed glow of all the galaxies in one spectral line, instead of detecting each galaxy
* refreshments from 3:30 pm in Astronomy 222. Colloquia are every Tuesday at 3:45 in this classroom. Go

</div>
</div>

<div style="font-size:0.7em; color:#666;">Photo: Illinois Astronomy directory, used to advertise her own talk. Listing: astro.illinois.edu/news-events/astrophysics-colloquium and calendars.illinois.edu (checked 2026-10-06).</div>

## Lab defenses

* one defense per student over the semester, on a lab or the midterm
* today: **Anuva Ghosalkar** and **Joseph Xu**, each on Lab 04, here up front
* Abha and I ask about your choices and your checks
* your notebook will be run in class - it must execute end to end, from a fresh kernel, with its data paths
* everyone else listens - these are the questions you will get too
* next defenses, Tue Oct 20: **Andrew Steves and KC Ciaccio**

## EXTRA: detailed balance, derived

* the fraction of a chain's steps at each $\theta$ is $p(\theta) = P(\theta \mid D)$. Why?
* $T(\theta' \mid \theta)$ is the transition probability of a step from $\theta$ to $\theta'$
* the probability of a point in the chain being at $\theta'$ (marginalizing over the possible immediately preceding points):
$$p(\theta') = \int p(\theta)\, T(\theta' \mid \theta)\, d\theta$$
* if this holds, a chain that is sampling $p$ keeps sampling $p$: $p$ is the chain's stationary distribution
* we want to have **detailed balance**:
$$p(\theta)\, T(\theta' \mid \theta) = p(\theta')\, T(\theta \mid \theta')$$
* integrate both sides over $\theta$; since the chain has to go somewhere, $\int T(\theta \mid \theta')\, d\theta = 1$, and you get the first equation back
* reversible: the probability of getting from $\theta$ to $\theta'$ is the same as from $\theta'$ to $\theta$

## EXTRA: the Metropolis-Hastings rule, derived

* we'll break the transition $T(\theta' \mid \theta)$ into two steps: a proposal $g(\theta' \mid \theta)$, and an acceptance probability $A(\theta' \mid \theta)$
* put $T = A\,g$ into detailed balance:
$$\frac{A(\theta' \mid \theta)}{A(\theta \mid \theta')} = \frac{p(\theta')\, g(\theta \mid \theta')}{p(\theta)\, g(\theta' \mid \theta)}$$
* pick the larger of the two $A$'s to be 1:
$$A(\theta' \mid \theta) = \min\left[1,\ \frac{p(\theta')\, g(\theta \mid \theta')}{p(\theta)\, g(\theta' \mid \theta)}\right]$$
* with this definition of $A$, detailed balance is automatically satisfied
* $P(D)$ cancels in $p(\theta')/p(\theta)$, so Metropolis never needs the evidence
* for two states, the Metropolis chain as a transition matrix (row = where you are, column = where you go next):
$$T = \begin{pmatrix} 0 & 1 \\ 1/4 & 3/4 \end{pmatrix}$$
* asymmetric proposal: multiply $R$ by $g(\theta \mid \theta')/g(\theta' \mid \theta)$, the Hastings correction. Leave it out and the chain is no longer sampling the posterior

## EXTRA: the Gelman-Rubin formula

* $m$ chains of $n$ steps each, burn-in removed. $\bar\theta_j$: chain $j$'s mean; $\bar\theta$: the global mean; $s_j^2$: chain $j$'s variance
$$B = \frac{n}{m-1}\sum_j \left(\bar\theta_j - \bar\theta\right)^2, \qquad W = \frac{1}{m}\sum_j s_j^2$$
* $B$: between chains (not the Bayes factor). $W$: within chains. $V$: the overall estimate for the variance of $\theta$
$$V = \frac{n-1}{n}\,W + \frac{1}{n}\,B, \qquad \hat R = \sqrt{\frac{V}{W}}$$

* `ArviZ`'s $\hat R$ splits each chain in half and uses the ranks of the samples instead of their values, so a few far-out samples can't dominate (Vehtari et al. 2021). It wants $\hat R < 1.01$; older texts use 1.1

<div style="font-size:0.8em; color:#666;">Gelman &amp; Rubin 1992, Statist. Sci. 7, 457; this form as in Gelman et al. 2013, <i>Bayesian Data Analysis</i>, 3rd ed.</div>

## EXTRA: HMC in 100 dimensions

* the same comparison for a Gaussian in 100 dimensions. Metropolis wanders slowly, while HMC jumps across the posterior at every iteration

<img src="images/neal2011_fig6_rwm_vs_hmc_100d.png" alt="Two trace plots over 1000 iterations for a 100-dimensional Gaussian: random-walk Metropolis on the left wanders slowly between -3 and 2, HMC on the right scatters independently across the full range every iteration" style="max-height:230px; display:block; margin:0 auto;">

<div style="font-size:0.7em; text-align:center; color:#666;">Neal 2011, in <i>Handbook of Markov Chain Monte Carlo</i> (arXiv:1206.1901), Fig. 6: one coordinate of a 100-D Gaussian, HMC with L = 150. To play with it: Chi Feng's <a href="https://chi-feng.github.io/mcmc-demo/app.html?algorithm=HamiltonianMC&target=banana">MCMC gallery</a>.</div>

**For smooth posteriors, HMC's advantage grows with the number of parameters.**